In [1]:
from collections import defaultdict
from collections import Counter
from ete3 import PhyloTree
from ete3 import Tree
import numpy as np
from tqdm.notebook import trange, tqdm
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import time
import csv
import copy
from glob import glob

#### Load gene families

In [2]:
exclSp=set(['Hofmia','Epibro'])

In [3]:
spTr=PhyloTree(open('metazoa-5.tre').read(), format=1)

In [21]:
print(spTr)


         /-Ephmul
      /-|
   /-|   \-Ampque
  |  |
  |   \-Osccar
  |
  |      /-Nemvec
  |   /-|
  |  |  |   /-Sanmal
  |  |   \-|
  |  |      \-Rhoesc
  |  |
  |  |            /-Strpur
--|  |         /-|
  |  |      /-|   \-Astrub
  |  |     |  |
  |  |   /-|   \-Sackow
  |  |  |  |
  |  |  |  |   /-Braflo
  |  |  |   \-|
  |  |  |     |   /-Lepocu
  |  |  |      \-|
  |  |  |         \-Musmus
  |  |  |
   \-|  |                              /-Tricas
     |  |                           /-|
     |  |                        /-|   \-Dromel
     |  |                       |  |
     |  |                     /-|   \-Clodip
     |  |                    |  |
     |  |                    |  |   /-Dapmag
     |  |                    |   \-|
     |  |                  /-|      \-Ampamp
     |  |                 |  |
     |  |                 |  |      /-Hyaazt
     |  |                 |  |   /-|
     |  |                 |   \-|   \-Penmon
     |  |               /-|     |
     |  |        

In [5]:
tll=set([l.name for l in spTr])

In [ ]:
tll

In [6]:
Fams,species={},[]
for line in open('orthologous_groups_0623.txt'):
    if line.startswith("#"): continue
    fid,glt=line.strip().split('\t')
    gbs=defaultdict(list)
    for g in glt.split():
        spc,gid=g.split('_',1)
        if spc in tll:
            gbs[spc].append(gid)
    species.extend(gbs.keys())
    if len(gbs)==0: continue 
    Fams[fid]=gbs
        #if i==5: break


In [ ]:
print(list(set(species)))

In [ ]:
print(list(Fams.items())[0])
print(species)

Creates non-redundant species set

Load species tree

In [5]:
spT=PhyloTree(open('metazoa-4.tre').read(), format=1)

In [ ]:
print(spT)

In [18]:
spTl=PhyloTree(open('metazoa-5-legs.tre').read(), format=1)

### functional annotation 

In [28]:
panther={r[0].split('_',1)[1]:r[1] for r in csv.reader(open('panther/Pricau.pthr'),delimiter='\t')} 

In [25]:
GOs=defaultdict(set)
PFAMs=defaultdict(set)
for l in open('go_files/PriCau_v1_ahfp.eggnog.emapper.annotations'):
    if l.startswith('#'): continue
    r=l.strip().split('\t')
    gid=r[0].split('.',1)[0]
    #print(gid)
    if not r[9]=='-': GOs[gid].update(r[9].split(','))
    if not r[-1]=='-': PFAMs[gid].update(r[-1].split(','))

In [26]:
print(list(PFAMs.items())[0])

('PCA23581', {'DUF1759', 'Peptidase_A17', 'rve'})


In [27]:
pantherNames={r[0].replace('.mag.mod','').replace('.mod','').replace('.',':'):r[1] for r in csv.reader(open('names.tab'),delimiter='\t')}

### Determine phylostrata of genes 

In [14]:
PriapPhylo=defaultdict(int)
tor=defaultdict(int)
phygenes={}
for fid in Fams:
    gbs=Fams[fid]
    spset=[s for s,gbs in gbs.items() if len(gbs)>0]
    phtyp=spT.get_common_ancestor(spset) if len(spset)>1 else spT.search_nodes(name=spset[0])
    tor[phtyp.name]+=1
    for g in gbs['Pricau']:
        PriapPhylo[phtyp.name]+=1
        phygenes[g]=(phtyp.name,fid)

In [7]:
print(PriapPhylo,sum(PriapPhylo.values()))

defaultdict(<class 'int'>, {'Metazoa': 20265, 'Eumetazoa': 2839, 'Bilateria': 1431, 'Protostomia': 501, 'Priapulida': 624, 'Ecdysozoa': 126}) 25786


In [15]:
with open('Pcau_phylostrata.txt','w') as out:
    for g,p in phygenes.items():
        out.write("{}\t{}\t{}\n".format(g,p[0],p[1]))

In [8]:
print(tor)

defaultdict(<class 'int'>, {'Metazoa': 9188, 'Eumetazoa': 1989, 'Bilateria': 1477, 'Protostomia': 1036, 'Spiralia': 456, 'Lophotrochozoa': 83, 'Spiralia_A': 70, 'Spiralia_B': 183, 'Demospongia': 226, 'Porifera': 96, 'Vermizoa': 9, 'Priapulida': 364, 'Mandibulata': 67, 'Ecdysozoa': 102, 'Lopho_A': 26, 'Ecdy_A': 73, 'Cnidaria': 241, 'Anthozoa': 345, 'Nematoda': 302, 'Deuterostomia': 72, 'Echinodermata': 132, 'Ambulacraria': 83, 'Rhbdidita': 150, 'Nem_1': 47, 'Chromadorea': 600, 'Xiphosura': 302, 'Chordata': 36, 'Mollusca': 156, 'Conchifera': 58, 'Pleistomollusca': 41, 'Xenambulacraria': 6, 'Annelida': 28, 'Euarthopoda': 133, 'Panarthropoda': 37, 'Pancrustacea': 52, 'Vertebrata': 539, 'Annelid_A': 56, 'Medusozoa': 662, 'Lophophorata': 16, 'Millipede': 219, 'Myriapoda': 25, 'Tardigrada': 1234, 'Holometabola': 203})


## Testing family expansions

In [82]:
#only testing species expansion
def fishExp(gbs,tbs):
    cbs=dict((sp,len(gs)) for sp,gs in gbs.items())
    med=np.median(list(cbs.values()))
    #print(cbs,med)
    spval={}
    for sp,st in cbs.items():
        nis=sum(cbs.values())-st
        nit=sum(tbs.values())-tbs[sp]-sum(cbs.values())+st
        odds,pval=stats.fisher_exact([[st, tbs[sp]-st], [nis, nit]],alternative='greater')
        nmed=np.median([cbs[ssp] for ssp in cbs if not ssp==sp])
        tmed=np.median([tbs[ssp] for ssp in tbs if not ssp==sp])
        m_odds,m_pval=stats.fisher_exact([[st, tbs[sp]-st], [nmed, tmed]],alternative='greater')
        #print(sp,pval,[st, tbs[sp]-st], [nis, nit],m_pval,[st, tbs[sp]-st], [nmed, tmed])
        spval[sp]=(cbs[sp],m_pval)

    return spval

calculate number of genes per species

In [83]:
tbs=defaultdict(int)
for fid,gbs in Fams.items():
    for sp in gbs:
        tbs[sp]+=len(gbs[sp])

In [84]:
toosmall=0
expRes=[]
for fid,gbs in tqdm(Fams.items()):
    cbs=dict((sp,len(gs)) for sp,gs in gbs.items())
    if len(cbs.keys())<3 or sum(cbs.values())<5:
        toosmall+=1
        continue
    try:
        spval=fishExp(gbs,tbs)
        expRes.append((fid,spval))
    except:
        print(fid,cbs)

In [98]:
print(len(Fams),len(expRes),toosmall)

21220 14224 6996


In [99]:
spFam,spPval=defaultdict(list),defaultdict(list)
for fid,spval in expRes:
    for sp in spval:
        ct,pval=spval[sp]
        spFam[sp].append(fid)                     
        spPval[sp].append(pval)

In [100]:
famEnrich=defaultdict(dict)
for sp in spPval:
    #signif_pvals = sidak(spPval[sp], alpha=0.05)
    #before_adj=['True' if p < 0.05  else 'False' for p in spPval[sp] ]
    #1signif_pvals = lsu(np.array(spPval[sp]), q=0.05) #this is benferroni-hochberg
    adj=multipletests(pvals=np.array(spPval[sp]), alpha=0.05, method="fdr_bh")
    print (sp,Counter(adj[0]))
    for fam,pth,pval in zip(spFam[sp],adj[0],adj[1]):
        famEnrich[fam][sp]=(pth,pval)
        


Notgen Counter({False: 9542, True: 73})
Ephmul Counter({False: 6395, True: 121})
Linana Counter({False: 10063, True: 20})
Fungia Counter({False: 9131, True: 117})
Melinc Counter({False: 5487, True: 82})
Ampque Counter({False: 7121, True: 142})
Pricau Counter({False: 14169, True: 55})
Braflo Counter({False: 9263, True: 61})
Pecmax Counter({False: 9767, True: 54})
Sackow Counter({False: 9445, True: 63})
Owefus Counter({False: 9770, True: 61})
Acagra Counter({False: 9594, True: 44})
Carrot Counter({False: 7688, True: 37})
Musmus Counter({False: 8560, True: 40})
Limpol Counter({False: 8005, True: 9})
Captel Counter({False: 9472, True: 66})
Sanmal Counter({False: 7922, True: 69})
Astrub Counter({False: 9153, True: 33})
Phoaur Counter({False: 9536, True: 19})
Lepocu Counter({False: 8529, True: 19})
Romcul Counter({False: 6207, True: 68})
Lotgig Counter({False: 9369, True: 39})
Helrob Counter({False: 7165, True: 59})
Rhoesc Counter({False: 8187, True: 31})
Bugner Counter({False: 7426, True: 3

In [70]:
DromelNames={e[0]:e[4] for e in csv.reader(open('Dromel_info.txt'),delimiter='\t')}

In [ ]:
print(DromelNames)

In [ ]:
with open('Orthogroups_enrich_Pc.txt','w') as out:
    for fid,gbs in Fams.items():
        cbs=dict((sp,len(gs)) for sp,gs in gbs.items())
        pvl=famEnrich[fid]
        drnm=[DromelNames.get(g,'-') for g in gbs.get('Dromel',[])]
        panth=set([pantherNames[panther[g]].lower() for g in gbs.get('Pricau',[]) if g in panther])
        print(panth,drnm)
        enriched= {s:p[1] for s,p in pvl.items() if p[0]==True}
        if len(enriched)==0: continue
        if not 'Pricau' in enriched: continue
        #print(enriched,drnm)

        outList=[fid,len(cbs),sum(cbs.values()),np.median(list(cbs.values())),','.join([s for s,p in enriched.items()]),
                 ','.join([str(cbs[s]) for s,p in enriched.items()]),','.join(drnm),','.join(['{:.2e}'.format(float(p)) for s,p in enriched.items()]),','.join(list(panth))]
        out.write('\t'.join(map(str,outList))+'\n')
        
    

## PANTHER

In [28]:
Pthr=defaultdict(dict)
for fp in glob("panther/*.pthr"):
    sp=fp.split('/')[1].replace('.pthr','')
    pps=defaultdict(list)
    #print(sp)
    for r in csv.reader(open(fp),delimiter='\t'):
        if not '_' in r[0]: 
            print(r)
        pps[r[1]].append(r[0].split('_',1)[1])
        #print(pps)
        #break
    for pth in pps:
        #pthr[pth]
        Pthr[pth][sp]=pps[pth]
        
    #print(list(pps.items())[0])
    #print(sp)

In [34]:
print(list(Pthr.items())[0])

('PTHR45935:SF9', {'Carrot': ['SINCROG011529', 'SINCROG011952', 'SINCROG003735', 'SINCROG020092', 'SINCROG010588', 'SINCROG007161', 'SINCROG003284', 'SINCROG008113', 'SINCROG016922', 'SINCROG012653', 'SINCROG010265', 'SINCROG010955', 'SINCROG018077', 'SINCROG010245', 'SINCROG005504', 'SINCROG012633', 'SINCROG000729', 'SINCROG014070', 'SINCROG001321', 'SINCROG018281', 'SINCROG005478', 'SINCROG013819', 'SINCROG014450', 'SINCROG011989', 'SINCROG016394', 'SINCROG017155', 'SINCROG009702', 'SINCROG025441', 'SINCROG006026', 'SINCROG011252', 'SINCROG008672', 'SINCROG004649', 'SINCROG007563', 'SINCROG013089', 'SINCROG011407', 'SINCROG015394', 'SINCROG004241', 'SINCROG013817', 'SINCROG024830', 'SINCROG003385', 'SINCROG006512', 'SINCROG015656', 'SINCROG003216', 'SINCROG015414', 'SINCROG002375', 'SINCROG021072', 'SINCROG020091', 'SINCROG003029', 'SINCROG015653', 'SINCROG006953', 'SINCROG002778', 'SINCROG013627', 'SINCROG025829', 'SINCROG010198', 'SINCROG005623', 'SINCROG020293', 'SINCROG011644', '

In [44]:
print(len(Pthr))

50938


In [6]:
origins=defaultdict(int)
losses=defaultdict(int)
idGainLoss={}
for fid,fam in tqdm(list(Pthr.items())):
    nspec,ndesc,oritax,lost=checkFam(fam,spT)
    idGainLoss[fid]=(nspec,ndesc,oritax,lost)
    origins[oritax]+=1
    for tax in lost:
        losses[tax]+=1
        

  0%|          | 0/50938 [00:00<?, ?it/s]

In [7]:
print(losses)
print(origins)

defaultdict(<class 'int'>, {'Ambulacraria': 1889, 'Chordata': 4568, 'Pancrustacea': 1819, 'Strmar': 5285, 'Triant': 3787, 'Tardigrada': 11894, 'Chromadorea': 1104, 'Adivag': 13698, 'Acagra': 5497, 'Pleistomollusca': 1911, 'Lophophorata': 5880, 'Vermizoa': 3667, 'Owefus': 9638, 'Medusozoa': 3985, 'Sackow': 3568, 'Vertebrata': 3926, 'Caeele': 2864, 'Helhol': 1822, 'Ramvar': 1793, 'Brumal': 4432, 'Romcul': 5140, 'Priapulida': 9849, 'Notgen': 3732, 'Helrob': 5524, 'Nemvec': 3565, 'Insecta': 2688, 'Dapmag': 3278, 'Crustacea': 3155, 'Myriapoda': 6239, 'Hypduj': 1349, 'Halspi': 3065, 'Bugner': 6332, 'Phoaur': 3151, 'Maclig': 4724, 'Ampque': 2482, 'Osccar': 4474, 'Cnidaria': 3589, 'Echinodermata': 2369, 'Dromel': 2069, 'Clodip': 3285, 'Ampamp': 2443, 'Hyaazt': 3059, 'Trispi': 6476, 'Annelid_A': 1107, 'Demospongia': 2468, 'Fungia': 1567, 'Hofmia': 8773, 'Braflo': 6588, 'B': 2671, 'Nematoda': 9619, 'Captel': 2623, 'Rhbdidita': 874, 'Linana': 1783, 'Rhoesc': 1785, 'Euraff': 5321, 'Tricor': 2992, 

In [8]:
spTl=copy.deepcopy(spT)
for node in spTl.traverse("postorder"):
    #print node.name,origins[node.name],losses[node.name]
    #node.add_features(origins=origins[node.name],losses=losses[node.name])
    #node.name="{0}_{1}_{2}".format(node.name,origins[node.name],losses[node.name])
    if node.is_leaf():
        node.name="{}_{}".format(node.name,losses[node.name])
    else:
        node.name="{}".format(losses[node.name])

In [11]:
with open('panther-losses.tre','w') as out:
    out.write(spTl.write(format=1))

## Gain and losses

In [ ]:
print(spT)


In [7]:
def checkFam(gbs,spTc):
    spset=[s for s,gbs in gbs.items() if len(gbs)>0]
    #print(phtyp.name)
    #phyloCt[phtyp.name]+=1
    #gbs[Ai]        
    spTd=copy.deepcopy(spTc)
    for leaf in spTd:
        pv='1' if leaf.name in gbs else '0'
        leaf.add_features(presence=pv)
    #print(spTc)
    phtyp=spTd.get_common_ancestor(spset) if len(spset)>1 else spTd.search_nodes(name=spset[0])[0]
    #print(phtyp)
    ndesc=len([l.name for l in phtyp.get_leaves()])
    lost=[node.name  for node in phtyp.get_monophyletic(values=['0'], target_attr="presence")]
    #print spTd.get_ascii(attributes=["name","presence"], show_internal=True)
    #for node in spTd.get_monophyletic(values=['0'], target_attr="presence"):
    #    print node.name
    #    print node.get_ascii(attributes=["presence", "name"], show_internal=False)
    return len(gbs.keys()),ndesc,phtyp.name,lost



In [115]:
checkFam(Fams['OG_8'],spT)

(41,
 50,
 'Metazoa',
 ['Clodip', 'B', 'Hyaazt', 'Euraff', 'Carrot', 'Triant', 'Trispi', 'Bugner'])

In [116]:
def checkFamCounts(fam,spTd):
    fid,cbs=fam
    spset=[s for s,n in cbs.items() if n>0]
    if len(spset)>1:
        phtyp=spT.get_common_ancestor(spset)
    else:
        phtyp=[l for l in spT.get_leaves() if l.name==spset[0]][0]
    #print phtyp.name
    #phyloCt[phtyp.name]+=1
    #gbs[Ai]
    ndesc=len([l.name for l in phtyp.get_leaves()])
    for leaf in spTd:
        pv='1' if leaf.name in spset else '0'
        leaf.add_features(presence=pv)
    lost=[node.name  for node in phtyp.get_monophyletic(values=['0'], target_attr="presence")]
    #print spTd.get_ascii(attributes=["name","presence"], show_internal=True)
    #for node in spTd.get_monophyletic(values=['0'], target_attr="presence"):
    #    print node.name
    #    print node.get_ascii(attributes=["presence", "name"], show_internal=False)
    return fid,len(spset),ndesc,phtyp.name,lost




In [ ]:
Fams['OG_8']

In [ ]:
for f in  Fams:
    gbs=Fams[f]
    if not 'Pricau' in gbs and 'Halspi' in gbs:
        print(f)

In [74]:
spset=[s for s,gbs in gbs.items() if len(gbs)>0]
spTc=copy.deepcopy(spT)
for leaf in spTc:
    pv='1' if leaf.name in gbs else '0'
    leaf.add_features(presence=pv)


In [ ]:
Fams['OG_2']

In [ ]:
print(spTc.get_ascii(attributes=["name","presence"], show_internal=True))

In [33]:
checkFam(Fams['OG_1'],spTr)

(44, 44, 'Metazoa', [])

In [ ]:
for s in spT: 
    print(s,s.name in set(species))

In [9]:
origins=defaultdict(int)
losses=defaultdict(int)
idGainLoss={}
for fid,gbs in Fams.items():
    try:
        nspec,ndesc,oritax,lost=checkFam(gbs,spTr)
    except:
        print(fid)
    idGainLoss[fid]=(nspec,ndesc,oritax,lost)
    origins[oritax]+=1
    for tax in lost:
        losses[tax]+=1
    
        

In [ ]:
checkFam(Fams['OG_7774'],spT)

In [ ]:
pc

In [32]:
from itertools import combinations
from itertools import product	

In [34]:
pc=defaultdict(int)
for f,r in tqdm(idGainLoss.items()):
    #print(f,r[3])
    tlist=[t for t in r[3] if not t in tll]
    for c1,c2 in product(tlist,repeat=2):
        #print(c1,c2)
        pc[(c1,c2)]+=1
        
    

  0%|          | 0/22688 [00:00<?, ?it/s]

In [35]:
with open('coloc_0623.txt','w') as out:
    out.write("cld1\tcld2\tNJLoss\n")
    for f in pc:
        ff=list(f)
        out.write("{}\t{}\t{}\n".format(ff[0],ff[1],pc[f]))

In [13]:
print(losses)
print(origins)

defaultdict(<class 'int'>, {'Melinc': 1299, 'Romcul': 1288, 'Captel': 1271, 'Cnidaria': 1172, 'Deuterostomia': 864, 'Ecdysozoa': 1059, 'Mollusca': 1511, 'Lophophorata': 1450, 'Notgen': 1964, 'Strpur': 545, 'Dromel': 902, 'Dapmag': 1233, 'Penmon': 1744, 'Helhol': 610, 'Tardigrada': 4922, 'Halspi': 928, 'Nem_1': 636, 'Lotgig': 1024, 'Medusozoa': 942, 'Ampamp': 1194, 'Hyaazt': 1017, 'Pecmax': 637, 'Ephmul': 1459, 'Vertebrata': 1315, 'Clodip': 949, 'Cru_A': 591, 'Tricor': 1413, 'Chelicerata': 2488, 'Priapulida': 1663, 'Acagra': 1146, 'Phoaur': 1176, 'Owefus': 1260, 'Nematoda': 4269, 'Carrot': 769, 'Trispi': 2203, 'Pricau': 637, 'Euarthopoda': 189, 'Braflo': 790, 'Euraff': 2348, 'Musmus': 367, 'Holometabola': 502, 'Ramvar': 379, 'Ampque': 714, 'Osccar': 1308, 'Tubcor': 1394, 'Brumal': 968, 'Xiphosura': 374, 'Insecta': 1001, 'Demospongia': 1185, 'Sanmal': 613, 'Caeele': 496, 'Echinodermata': 657, 'Annelida': 550, 'Rhoesc': 395, 'C': 1502, 'Myriapoda': 1436, 'Millipede': 484, 'Triant': 1298, 

In [14]:
spTi=copy.deepcopy(spT)
for node in spTi.traverse("postorder"):
    if node.is_leaf():
        node.name="{}_{}".format(node.name,origins[node.name])
    else:
        node.name="{}".format(origins[node.name])

NameError: name 'spT' is not defined

In [36]:
with open('metazoa-origins.tre','w') as out:
    out.write(spTi.write(format=1))

In [15]:
spTl=copy.deepcopy(spTr)
for node in spTl.traverse("postorder"):
    if node.is_leaf():
        node.name="{}_{}".format(node.name,losses[node.name])
    else:
        node.name="{}".format(losses[node.name])

In [16]:
spTo=copy.deepcopy(spTr)
for node in spTo.traverse("postorder"):
    if node.is_leaf():
        node.name="{}_{}".format(node.name,origins[node.name])
    else:
        node.name="{}".format(origins[node.name])

In [9]:
with open('metazoa-losses-tri.tre','w') as out:
    out.write(spTl.write(format=1))

In [18]:
print(spTo.write(format=1))

(((Ephmul_31:1,Ampque_42:1)234:1,Osccar_52:1)98:1,((Nemvec_344:1,(Sanmal_24:1,Rhoesc_21:1)697:1)172:1,((((Strpur_9:1,Astrub_6:1)132:1,Sackow_21:1)79:1,(Braflo_23:1,(Lepocu_9:1,Musmus_7:1)530:1)37:1)73:1,((((((((((Tricas_7:1,Dromel_3:1)88:1,Clodip_5:1)112:1,(Dapmag_4:1,Ampamp_21:1)4:1)34:1,((Hyaazt_34:1,Penmon_18:1)266:1,Euraff_31:1)22:1)204:1,((Helhol_30:1,Tricor_20:1)207:1,Strmar_8:1)20:1)226:1,((Limpol_11:1,Carrot_24:1)231:1,Triant_12:1)75:1)207:1,(Hypduj_14:1,Ramvar_13:1)1217:1)64:1,((((Caeele_8:1,Melinc_25:1)158:1,Brumal_2:1)599:1,Trispi_13:1)47:1,Romcul_31:1)311:1)161:1,((Pricau_38:1,Halspi_16:1)191:1,Tubcor_52:1)340:1)678:1,((Acagra_37:1,(Pecmax_23:1,Lotgig_41:1)51:1)150:1,((Phoaur_17:1,Linana_26:1)30:1,(Notgen_34:1,(Owefus_34:1,Captel_94:1)26:1)44:1)148:1)416:1)1048:1)1345:1)1493:1);


In [ ]:
print(spTi.get_ascii(show_internal=True))

In [ ]:
[idGainLoss.items()][0]

In [10]:
igl=[[s,v[0],v[1],v[2],','.join(v[3])] for s,v in idGainLoss.items()]

In [11]:
pcg={f:','.join([g for g in gbs.get('Pricau',[])]) for f,gbs in Fams.items()}

In [12]:
bfg={f:','.join([g for g in gbs.get('Braflo',[])]) for f,gbs in Fams.items()}

In [13]:
bfg={f:','.join([g for g in gbs.get('Braflo',[])]) for f,gbs in Fams.items()}

In [14]:
bfg={f:','.join([g for g in gbs.get('Braflo',[])]) for f,gbs in Fams.items()}

In [29]:
PcaGn={f:','.join([g for g in gbs.get('Pricau',[])]) for f,gbs in Fams.items()}

In [48]:
HsiGn={f:','.join([g for g in gbs.get('Halspi',[])]) for f,gbs in Fams.items()}

In [57]:
PcauP={f:','.join(list(set([panther[g] for g in gbs.get('Pricau',[]) if g in panther]))) for f,gbs in Fams.items()}

In [38]:
PcauPth={f:','.join(list(set([pantherNames[panther[g]] for g in gbs.get('Pricau',[]) if g in panther]))) for f,gbs in Fams.items()}

In [67]:
pfamP={f:','.join([p for g in gbs['Pricau'] if 'Pricau' in gbs for p in PFAMs[g]if g in PFAMs ]) for f,gbs in Fams.items()}

In [72]:
droso={f:','.join([DromelNames.get(g,g) for g in gbs['Dromel'] if 'Dromel' in gbs]) for f,gbs in Fams.items()}

In [82]:
dups={f:','.join(d for f,d in dupNodes.items}

TypeError: 'builtin_function_or_method' object is not iterable

In [ ]:
dupNodes

In [44]:
fidGL["Pca_genes"] = pd.Series(PcaGn)

In [49]:
fidGL["Hsp_genes"] = pd.Series(HsiGn)

In [58]:
fidGL["Panther_nm"] = pd.Series(PcauPth)

In [59]:
fidGL["Panther"] = pd.Series(PcauP)

In [74]:
fidGL["Dmel_genes"] = pd.Series(droso)

In [68]:
fidGL["PFAM"] = pd.Series(pfamP)

In [84]:
fidGL["Duplications"] = pd.Series(dupNodes)

In [18]:
fidGL=pd.DataFrame(igl, columns = ['FID', 'NbSpecies','NbDesc','Strata','Losses'])  

In [16]:
import pandas as pd

In [85]:
fidGL

,NbSpecies,NbDesc,Strata,Losses,Bfl_gn,Pca_genes,Panther,Hsp_genes,Panther_nm,PFAM,Dmel_genes,Duplications
FID,,,,,,,,,,,,
OG_1,44,44,Metazoa,,NaN,"PCA32878,PCA34374",PTHR10572,ScKMIYO_104.g5894,3-HYDROXY-3-METHYLGLUTARYL-COENZYME A REDUCTASE,"HMG-CoA_red,Sterol-sensing,HMG-CoA_red,Sterol-...",Hmgcr,"Melinc,Priapulida,Xiphosura,Notgen,Tricas,Limp..."
OG_2,41,44,Metazoa,"Melinc,Romcul,Captel",NaN,PCA26658,PTHR21681:SF0,ScKMIYO_436.g31347,EUKARYOTIC TRANSLATION INITIATION FACTOR 3 SUB...,eIF3_subunit,eIF3j,"Musmus,Osccar,Sanmal,Limpol"
OG_3,4,44,Metazoa,"Cnidaria,Deuterostomia,Ecdysozoa,Mollusca,Loph...",NaN,,,,,,,"Owefus,Osccar"
OG_4,36,44,Metazoa,"Strpur,Dromel,Dapmag,Penmon,Helhol,Tardigrada,...",NaN,PCA08382,PTHR11006,,PROTEIN ARGININE N-METHYLTRANSFERASE,PrmA,,"Melinc,Clodip,Notgen,Ephmul,Romcul"
OG_5,39,44,Metazoa,"Nem_1,Lotgig",NaN,PCA21558,PTHR45809:SF4,ScKMIYO_426.g25366,PHOSDUCIN-LIKE PROTEIN 3,Phosducin,viaf,"Linana,Notgen,Limpol,Captel,Dapmag,Vertebrata"
...,...,...,...,...,...,...,...,...,...,...,...,...
OG_22804,1,1,Melinc,,NaN,,,,,,,NaN
OG_22805,1,1,Ampque,,NaN,,,,,,,NaN
OG_22806,2,2,Rhbdidita,,NaN,,,,,,,NaN


In [43]:
fidGL=fidGL.set_index('FID')

In [86]:
fidGL.to_csv('ortho0623_GainLoss.txt',sep='\t')

In [ ]:
pcg

In [ ]:
with open('Orthogroups_gain_loss.txt','w') as out:
    outNames=['fid','n_sp','n_dsp','n_g','origin','losses','duplics','Pca_g','Hsi_g','Dme_g','panther','pfam']
    out.write('\t'.join(outNames)+'\n')     #
    for fid,gbs in Fams.items():
        cbs=dict((sp,len(gs)) for sp,gs in gbs.items())
        pvl=famEnrich[fid]
        drnm=[DromelNames.get(g,'-') for g in gbs.get('Dromel',[])]
        pcg=[g for g in gbs.get('Pricau',[])]
        hsg=[g for g in gbs.get('Halspi',[])]
        panth=set([pantherNames[panther[g]].lower() for g in gbs.get('Pricau',[]) if g in panther])
        pfam=sum([list(PFAMs[g]) for g in gbs.get('Pricau',[]) if g in PFAMs],[])
        #enriched= {s:p[1] for s,p in pvl.items() if p[0]==True}
        #if len(enriched)==0: continue
        #if not 'Pricau' in enriched: continue
        #print(enriched,drnm)
        #dupNodes
        #outNames=['fid','n_sp','n_dsp','n_g','origin','losses','duplics','Pca_g','Hsi_g',dmel','panther','pfam']]
        nspec,ndesc,oritax,lost=idGainLoss[fid]

        #print(len(cbs),sum(cbs.values()),nspec,ndesc,oritax,lost)        
        dups=dupNodes.get(fid,'-')
        if 'Ecdyzozoa' in lost:
            print(outList)
        outList=[fid,len(cbs),ndesc,sum(cbs.values()),oritax,','.join(lost),dups,','.join(pcg),','.join(hsg),','.join(drnm),','.join(list(panth)),','.join(pfam)]
        #print(outList)
        out.write('\t'.join(map(str,outList))+'\n')
        #break
    

## Duplications

In [76]:
def decoTree(t,outtax=['Nemvec','Auraur']):
    for l in t:
        sp,gid=l.name.split('_',1)
        gid=gid.split('.')[0]
        l.add_features(species=sp,gid=gid)
    #first, reset rooting with mid-point rooting
    R = t.get_midpoint_outgroup()
    t.set_outgroup(R)
    #then root with deuterostome ancestor
    sot=[l for l in t if l.species in outtax] 
    if len(sot)>0:
        anc=t.get_common_ancestor(sot)
        if not anc==t:
            t.set_outgroup(anc)
    return t

In [77]:
def age(node,sptr):
    tsp=list(set([lf.species for lf in node]))
    if len(tsp)==1:
        return tsp[0]
    else:
        tanc=sptr.get_common_ancestor(tsp).name
        return tanc

In [ ]:
print(spT)

In [78]:
rTrees={}
nclgbfl=[]
strees={}
nexcl=0
for l in open('Priap_0124_nhx.txt'):
    og,ts=l.strip().split(':',1)
    #og=o.split('/')[-1].rsplit('_',1)[0]
    #print(og)
    strees[og]=ts
    t=Tree(ts, format=1)
    try:
        t=decoTree(t,outtax=['Ephmul','Ampque','Osccar'])
        spl=list(set([l.species for l in t]))
    #print(spl)
        if len(spl)<3:
            nexcl+=1
            continue
        rTrees[og]=t
    except:
        print("troubles with {}".format(og))

computing patristic distance for a number of taxa 

In [9]:
ft=rTrees['OG_10001']

In [9]:
subsp=['Tricor','Romcul','Ramvar','Pricau','Owefus','Braflo']

In [ ]:
print(ft)

In [ ]:
nlist=[n for no in ft.traverse("postorder")]


In [8]:
olst=[]
for l in open('Priap_0124_nhx.txt'):
    og,ts=l.strip().split(':',1)
    #og=o.split('/')[-1].rsplit('_',1)[0]
    #print(og)
    t=Tree(ts, format=1)
    pdists=defaultdict(list)
    try:
        t=decoTree(t,outtax=['Ephmul','Ampque','Osccar'])
        spl=list(set([l.species for l in t]))
        tr=t.get_tree_root()
        for s in spl:
            for l in t.search_nodes(species=s):
                pdists[s].append(tr.get_distance(l))
            
    except:
        print("troubles with {}".format(og))
    adst=sum(list(pdists.values()),[])
    #sdist=
    smed=[np.median(pdists[s]) for s in subsp]
    igl=idGainLoss[og]
    olst.append([og,np.median(adst),np.mean(adst),np.median(smed)]+smed+[igl[0],igl[1],igl[2],','.join(igl[3])])

    #break


NameError: name 'subsp' is not defined

In [113]:
import pandas as pd

In [114]:
pdlf = pd.DataFrame(olst, columns = ['FID', 'med','mean','smed','Tricor','Romcul','Ramvar','Pricau','Owefus','Braflo','ntax','ndsc','orig','loss']) 

In [117]:
pdlf.to_csv('fams_pat_gl.tsv',sep='\t',index=False)

In [107]:
idGainLoss['OG_10001']

(27,
 47,
 'Eumetazoa',
 ['Medusozoa',
  'Clodip',
  'B',
  'Crustacea',
  'Helhol',
  'Chelicerata',
  'Nematoda',
  'Priapulida'])

In [97]:
np.median(sum(list(pdists.values()),[]))
np.mean(sum(list(pdists.values()),[]))

1.6759620000000002

In [31]:
R = ft.get_midpoint_outgroup()
#ft.set_outgroup(R)
for sp in subsp:
    sublf=[l for l in ft if l.species==sp] 
    subdt=[R.get_distance(l) for l in sublf]
    print(sp,sublf,subdt)

Tricas [Tree node 'Tricas_TC002096' (0x126614bb)] [7.974651000000001]
Tricor [Tree node 'Tricor_Tco_002620' (0x12661497)] [7.897010000000001]
Romcul [Tree node 'Romcul_g27558' (0x1266154e)] [8.287548000000001]
Pricau [Tree node 'Pricau_PCA29493' (0x126614f1)] [8.254807]
Owefus [Tree node 'Owefus_OFUSG14808' (0x12661569)] [8.126134]
Pecmax [Tree node 'Pecmax_LOC117341984' (0x12661563)] [8.198877]
Braflo [Tree node 'Braflo_BL108091' (0x12661473)] [10.380491999999998]
Strpur [Tree node 'Strpur_LOC753648' (0x12661467)] [8.727523999999997]


In [26]:
print([l.name for l in ft if l.species=='Tricas'])
print([l.name for l in ft if l.species=='Pricau'])
print([l.name for l in ft if l.species=='Tricor'])
print([l.name for l in ft if l.species=='Braflo'])


['Tricas_TC034512']
['Pricau_PCA30836', 'Pricau_PCA27598', 'Pricau_PCA28601']
['Tricor_Tco_009310']
['Braflo_BL139591']


In [ ]:
print(ft)

In [79]:
dupNodes={}
excldna=0
totdups=defaultdict(int)
p_dup=[]
for f,st in tqdm(rTrees.items()):
    sta=age(st,spTr
           )
    #print('tree age:',sta)
    dupnds=[sst for sst in st.search_nodes(D='Y')]
    for nd in dupnds:
        if age(nd,spTr) in set(['Priapulida','Priapulimorpha']):
            p_dup.append((f,nd,age(nd,spTr)))
    ages=[age(n,spTr) for n in dupnds]
    for a in list(set(ages)):
        if not a==sta:
            totdups[a]+=1
        else:
            excldna+=1
    dupNodes[f]=','.join(list(set(ages)))
    #print(f,dupnds,ages)

  0%|          | 0/11895 [00:00<?, ?it/s]

In [80]:
dupNodes

{'OG_10001': 'Ramvar,Notgen,Strpur',
 'OG_10003': 'Trispi,Linana,Mandibulata,Euraff,Triant,Limpol,Dapmag,Dromel,Pricau,Melinc,Xiphosura,Ecdy_A,Lepocu,Hypduj,Hyaazt,Osccar,Notgen,Carrot,Tricas,Penmon,Ampque,Vertebrata',
 'OG_10004': 'Linana,Ramvar,Notgen,Carrot,Hypduj,Penmon,Ampamp,Caeele,Romcul,Tardigrada',
 'OG_10007': 'Linana,Notgen,Xiphosura,Carrot,Captel,Metazoa,Ampamp,Vertebrata,Romcul',
 'OG_10008': 'Linana,Notgen,Strmar,Euraff,Tubcor,Pricau,Ampque,Tardigrada',
 'OG_1000': 'Melinc,Osccar,Notgen,Hypduj,Xiphosura,Penmon,Captel,Ampque',
 'OG_10010': 'Melinc,Priapulida,Penmon,Hypduj,Triant,Ampque,Tardigrada',
 'OG_10013': 'Melinc,Linana,Xiphosura,Carrot,Triant,Limpol,Ampamp',
 'OG_10016': 'Trispi,Pecmax,Strpur,Linana,Porifera,Astrub,Sackow,Triant,Limpol,Mollusca,Echinodermata,Ephmul,Tardigrada,Pricau,Melinc,Priapulida,Xiphosura,Halspi,Ampamp,Nematoda,Tubcor,Romcul,Bilateria,Cru_A,Acagra,Hypduj,Captel,Phoaur,Lophotrochozoa,Strmar,Chelicerata,Osccar,Spiralia_B,Notgen,Penmon,Carrot,Lotg

In [ ]:
p_dup

In [12]:
with open('priap_dups_r.txt','w') as out: 
    for f,n,s in p_dup:
        print(f,s)
        #for l in n.traverse("postorder"):
        #    if age(l,spT)=='Priapulida' and not l.D=='Y':
        #        print(l)
        
        pg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Pricau")]
        hg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Halspi")]
        print(pg,hg)
        if len(pg)==2 and len(hg)==2:
            out.write("{}\t{}\t{}\n".format(f,','.join(pg),','.join(hg)))
        break

OG_10010 Priapulida
['PCA30151', 'PCA18872'] ['ScKMIYO_1.g1995', 'ScKMIYO_86.g38027']


In [ ]:
with open('priap_dups.txt','w') as out: 
    for f,n,s in p_dup:
        print(f,s)
        #for l in n.traverse("postorder"):
        #    if age(l,spT)=='Priapulida' and not l.D=='Y':
        #        print(l)
        
        pg,hg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Pricau")],[l.name.split('_',1)[1] for l in n.search_nodes(species="Halspi")]
        #print(pg,hg)
        if len(pg)==2 and len(hg)==2:
            out.write("{}\t{}\t{}\n".format(f,','.join(pg),','.join(hg)))
        #break

In [27]:
PcauPos={r[13]:(r[0],r[1],r[2]) for r in csv.reader(open("../Architecture/PriCau_v1_ahfp.bed"),delimiter='\t')}

In [ ]:
print(list(test.items())[0])

In [ ]:
,"#E6AB02","#A6761D","#8DD3C7","#666666","#FF7F00","#FDBF6F","#BEBADA","#ffd92f"

In [26]:
# pas le bon
equiv={frozenset(('chr1','chr2')):'"#6A3D9A"',
       frozenset(('chr5','chr6')):'"#A6761D"',
       frozenset(('chr7','chr10')):'"#8DD3C7"',
       frozenset(('chr9','chr13')):'"#666666"',
       frozenset(('chr1','chr3')):'"#FF7F00"',
       frozenset(('chr3','chr8')):'"#FDBF6F"',
       frozenset(('chr6','chr12')):'"#FDBF6F"',
       frozenset(('chr7','chr10')):'"#FDBF6F"',
       
      
      
      
      }

In [28]:
equiv={frozenset(('chr1','chr2')):'A1+O2', # also in strigamia
       frozenset(('chr3','chr6')):'A2',
       frozenset(('chr1','chr9')):'B1',
       #frozenset(('chr3','chr11')):'B2', # chr7 could alsl be included
       #frozenset(('chr1','chr2')):'B3', not B3 duplication (!)
       frozenset(('chr5','chr8')):'C1+P',
       frozenset(('chr5','chr6')):'C2+I',
       frozenset(('chr11','chr3')):'B2+D+F+J2', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr8','chr3')):'F', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr8','chr11')):'F', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr3','chr7')):'B2', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr11','chr7')):'B2', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr9','chr13')):'E',
       #frozenset(('chr3','chr8')):'F', #also chr11
       frozenset(('chr7','chr10')):'G+O1',
       frozenset(('chr4','chr8')):'H',
       frozenset(('chr1','chr11')):'J1',
       #frozenset(('chr3','chr11')):'J2', # also chr9
       frozenset(('chr6','chr12')):'K',
       frozenset(('chr2','chr7')):'L',
       frozenset(('chr4','chr5')):'M',
       frozenset(('chr2','chr4')):'N',
       #frozenset(('chr7','chr10')):'G+O1',
       #frozenset(('chr1','chr2')):'O2',
       #frozenset(('chr5','chr8')):'P',
       frozenset(('chr6','chr14')):'Q',
       #frozenset(('chr7','chr10')):'R', not duplicated detected
       
      
      
      
      }

In [ ]:
hs_eq={frozenset(('ScKMIYO_86','ScKMIYO_86')):'A1+O2', # also in strigamia
       frozenset(('chr3','chr6')):'A2',
       frozenset(('chr1','chr9')):'B1',
       #frozenset(('chr3','chr11')):'B2', # chr7 could alsl be included
       #frozenset(('chr1','chr2')):'B3', not B3 duplication (!)
       frozenset(('chr5','chr8')):'C1+P',
       frozenset(('chr5','chr6')):'C2+I',
       frozenset(('chr11','chr3')):'B2+D+F+J2', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr8','chr3')):'F', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr8','chr11')):'F', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr3','chr7')):'B2', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr11','chr7')):'B2', # also chr1  # in strigamia: B2+D+J2
       frozenset(('chr9','chr13')):'E',
       #frozenset(('chr3','chr8')):'F', #also chr11
       frozenset(('chr7','chr10')):'G+O1',
       frozenset(('chr4','chr8')):'H',
       frozenset(('chr1','chr11')):'J1',
       #frozenset(('chr3','chr11')):'J2', # also chr9
       frozenset(('chr6','chr12')):'K',
       frozenset(('chr2','chr7')):'L',
       frozenset(('chr4','chr5')):'M',
       frozenset(('chr2','chr4')):'N',
       #frozenset(('chr7','chr10')):'G+O1',
       #frozenset(('chr1','chr2')):'O2',
       #frozenset(('chr5','chr8')):'P',
       frozenset(('chr6','chr14')):'Q',
       #frozenset(('chr7','chr10')):'R', not duplicated detected

In [19]:
equiv.values()

dict_values(['A1+O2', 'A2', 'B1', 'C1+P', 'C2+I', 'B2+D+F+J2', 'F', 'F', 'B2', 'B2', 'E', 'G+O1', 'H', 'J1', 'K', 'L', 'M', 'N', 'Q'])

In [75]:
equiv[frozenset(('chr2','chr1'))]

'"#6A3D9A"'

In [142]:
print(len(p_dup))

1826


In [49]:
from itertools import combinations
import pandas as pd

In [68]:
i=0
adpc=set()
CtCd=defaultdict(int)
dout=[]
pout=[]
psp=['Pricau','Halspi','Tubcor']
for f,n,s in p_dup:
    l=[l.name for sn in n.children[0] for l in sn]
    r=[l.name for sn in n.children[1] for l in sn]
    pcg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Pricau")]
    #hsg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Halspi")]
    #tcg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Tubcor")]
    ch_pc=frozenset([PcauPos[g][0] for g in pcg])
    #print(n)
    #dpc,hpc=set(),set()
    ld = defaultdict(list, {sp.split('_')[0]: [g.split('_',1)[1] for g in l if g.startswith(sp.split('_')[0])] for sp in set(sg.split('_')[0] for sg in l)})
    rd = defaultdict(list, {sp.split('_')[0]: [g.split('_',1)[1] for g in r if g.startswith(sp.split('_')[0])] for sp in set(sg.split('_')[0] for sg in r)})
    #print(ld,rd)
    synt=equiv.get(ch_pc,'NA')
    if not synt=='NA': CtCd['Pc valid pairs']+=1
    #print(f,s,ld,rd,synt)
    pcc=','.join(list(ch_pc))
    pout.append([f,s,','.join(ld['Pricau']),','.join(ld['Halspi']),','.join(rd['Pricau']),','.join(rd['Halspi']),pcc,synt])         

    #adpc.update(dpc)            
    #print(f,s,len(pcg),len(hsg),len(tcg),pcg,ch_pc,equiv.get(ch_pc,'NA'))
    #dps='NA' if len(dpc)==0 else ','.join(list(dpc))
    #
    #dhs='NA' if len(hsg)==0 else ','.join(list(hsg))
    #oL=[f,s,len(pcg),len(hsg),len(tcg),dps,dhs,','.join(list(ch_pc)),equiv.get(ch_pc,'NA')]
    #dout.append(oL)
    i+=1
    #if i==50: break
            #print(f,pg)
        #break

print(i,CtCd)

1826 defaultdict(<class 'int'>, {'Pc valid pairs': 530})


In [69]:
pd.DataFrame(pout, columns = ['FID', 'Age','pcg1','hsg1','pcg2','hsg2','Pc_chr','ALG']).to_csv('Pca_dup_alg.txt',sep='\t',index=False)

In [29]:
i=0
adpc=set()
CtCd=defaultdict(int)
dout=[]
pout=[]
psp=['Pricau','Halspi','Tubcor']
for f,n,s in p_dup:
    print(f)
    l=[l.name for sn in n.children[0] for l in sn]
    r=[l.name for sn in n.children[1] for l in sn]
    print(l,r)
    
    pcg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Pricau")]
    hsg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Halspi")]
    tcg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Tubcor")]
    ch_pc=frozenset([PcauPos[g][0] for g in pcg])
    #print(n)
    dpc,hpc=set(),set()
    
    if not equiv.get(ch_pc,'NA')=='NA': 
        CtCd['Pc valid pairs']+=1
        for g0,g1 in combinations(pcg, 2):
            c0,c1=PcauPos[g0],PcauPos[g1]
            if c0[0]!=c1[0]:
                pout.append([f,s,g0,c0[0],c0[1],g1,c1[0],c1[1],equiv.get(ch_pc,'NA')])
                dpc.update([g0,g1])

                

    adpc.update(dpc)            
    #print(f,s,len(pcg),len(hsg),len(tcg),pcg,ch_pc,equiv.get(ch_pc,'NA'))
    dps='NA' if len(dpc)==0 else ','.join(list(dpc))
    #
    dhs='NA' if len(hsg)==0 else ','.join(list(hsg))
    oL=[f,s,len(pcg),len(hsg),len(tcg),dps,dhs,','.join(list(ch_pc)),equiv.get(ch_pc,'NA')]
    dout.append(oL)
    i+=1
    if i==50: break
            #print(f,pg)
        #break

print(i,CtCd)

OG_10010
['Halspi_ScKMIYO_1.g1995', 'Pricau_PCA30151', 'Tubcor_g20937.t1'] ['Halspi_ScKMIYO_86.g38027', 'Pricau_PCA18872', 'Tubcor_g8803.t1']


NameError: name 'combinations' is not defined

In [23]:
import pandas as pd

In [24]:
pd.DataFrame(pout, columns = ['FID', 'Age','g1','chr1','pos1','g2','chr2','pos2','ALG']).to_csv('Pca_dup_alg.txt',sep='\t',index=False)

In [29]:
dupp=pd.DataFrame(dout, columns = ['FID', 'Age','nPca','nHsi','nTcor','Pca_g','Hsp_g','Pca_chr','ALG'])

In [32]:
dupp

,FID,Age,nPca,nHsi,nTcor,Pca_g,Hsp_g,Pca_chr,ALG
0,OG_10010,Priapulida,2,2,2,"PCA18872,PCA30151","ScKMIYO_1.g1995,ScKMIYO_86.g38027","chr2,chr1",A1+O2
1,OG_10016,Priapulida,7,4,3,"PCA11640,PCA11995,PCA04043,PCA03101,PCA11670,P...","ScKMIYO_414.g23178,ScKMIYO_414.g23180,ScKMIYO_...","chr9,chr13",E
2,OG_10016,Priapulimorpha,2,2,0,NA,"ScKMIYO_414.g23178,ScKMIYO_414.g23180",chr9,NA
3,OG_1003,Priapulimorpha,31,2,0,NA,"ScKMIYO_96.g39449,ScKMIYO_104.g6493","chr9,chr2,chr8,chr12,chr13,chr1,scaf10,chr11,c...",NA
4,OG_1005,Priapulida,4,0,4,NA,NA,"chr12,chr8",NA
...,...,...,...,...,...,...,...,...,...
1821,OG_9941,Priapulida,2,2,2,"PCA15493,PCA29842","ScKMIYO_104.g4126,ScKMIYO_1.g3635","chr2,chr7",L
1822,OG_9959,Priapulimorpha,2,2,0,NA,"ScKMIYO_104.g4280,ScKMIYO_414.g22943","chr9,chr5",NA
1823,OG_9970,Priapulida,4,3,2,NA,"ScKMIYO_412.g21950,ScKMIYO_104.g5848,ScKMIYO_4...",chr7,NA
1824,OG_9970,Priapulimorpha,2,3,0,NA,"ScKMIYO_412.g21950,ScKMIYO_104.g5848,ScKMIYO_4...",chr7,NA


In [31]:
dupp.to_csv('Priap_ct_dup_e.txt',sep='\t',index=False)

In [211]:
with open('Pc_dup_raw.txt','w') as out:
    out.write('\n'.join(adpc))

In [156]:
#pc_dp=defaultdict(set)
sc,dc=0,0
with open('pcau_dups_pss.txt','w') as out: 
    for f,n,s in p_dup:
        pg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Pricau")]
        #print(pg)
        if len(pg)==2 and s=='Priapulida':
            p0,p1=PcauPos[pg[0]],PcauPos[pg[1]]
            s0,s1,c0,c1=p0[1],p1[1],p0[0],p1[0]
            if c0==c1: 
                sc+=1
            else:
                dc+=1
                pcol=equiv.get(frozenset((c0,c1)),'NA')
                #print(s0,s1,c0,c0,c1,pcol)
                out.write("{}\t{}\t{}\t{}\t{}\t{}\t{}\t{}\n".format(f,pg[0],c0,s0,pg[1],c1,s1,pcol))
            #print(f,pg)
        #break
print(sc,dc)

198 264


In [128]:
sc

198

In [ ]:
with open('pcau_dups.txt','w') as out: 
    for f,n in p_dup:
        #print(f,n)
        #for l in n.traverse("postorder"):
        #    if age(l,spT)=='Priapulida' and not l.D=='Y':
        #        print(l)
        
        #pg,hg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Pricau")],[l.name.split('_',1)[1] for l in n.search_nodes(species="Halspi")]
        pg=[l.name.split('_',1)[1] for l in n.search_nodes(species="Pricau")]

        print(f,pg)
        if len(pg)>=2 and len(hg)==2:
            out.write("{}\t{}\t{}\n".format(f,','.join(pg),','.join(hg)))
        #break

In [ ]:
for f,ld in dupNodes.items():
    if 'Priapulida' in ld:
        print(f)

In [10]:
print(excldna)

4661


In [ ]:
for f in dupNodes:
    if 'Spiralia' in dupNodes[f]:
        print(f)

In [ ]:
print(rTrees['OG_10002'].get_ascii(attributes=["name", "D"],show_internal=True))

In [ ]:
print(dupNodes)

In [55]:
print(totdups)

defaultdict(<class 'int'>, {'Strpur': 2781, 'Notgen': 6318, 'Ramvar': 489, 'Trispi': 1081, 'Triant': 1482, 'Mandibulata': 185, 'Ecdy_A': 284, 'Tricas': 629, 'Euraff': 1561, 'Limpol': 2509, 'Dromel': 729, 'Hypduj': 793, 'Xiphosura': 2822, 'Hyaazt': 1012, 'Pricau': 2358, 'Linana': 5316, 'Carrot': 1371, 'Osccar': 2402, 'Vertebrata': 2544, 'Dapmag': 894, 'Melinc': 4914, 'Penmon': 2895, 'Lepocu': 779, 'Ampque': 2137, 'Caeele': 738, 'Tardigrada': 631, 'Romcul': 1705, 'Ampamp': 3193, 'Captel': 2196, 'Tubcor': 1866, 'Strmar': 1057, 'Priapulida': 933, 'Lotgig': 1243, 'Demospongia': 226, 'Pecmax': 1478, 'Acagra': 1220, 'Echinodermata': 282, 'Ephmul': 2535, 'Bilateria': 1076, 'Cru_A': 301, 'Spiralia': 726, 'Astrub': 823, 'Priapulimorpha': 443, 'Porifera': 122, 'Sackow': 1763, 'Protostomia': 609, 'Mollusca': 325, 'Chelicerata': 352, 'Nematoda': 414, 'Halspi': 987, 'Phoaur': 1745, 'Spiralia_B': 372, 'Owefus': 1108, 'Medusozoa': 475, 'Rhoesc': 643, 'Helhol': 1179, 'Nemvec': 816, 'Tricor': 696, 'Musm

In [68]:
spTd=copy.deepcopy(spTr)
for node in spTd.traverse("postorder"):
    #print(node.name,totdups[node.name])
    #node.add_features(origins=origins[node.name],losses=losses[node.name])
    if node.is_leaf():
        node.name="{}_{}".format(node.name,totdups[node.name])
    else: 
        node.name="{}".format(totdups[node.name])


In [69]:
with open('metazoa-duplicates.tre','w') as out:
    out.write(spTd.write(format=1))

In [ ]:
print(spTi.get_ascii(show_internal=True))
print(spTi.write(format=1))

In [213]:
### Post GO 

In [215]:
pcNm={r[0]:r[1] for r in csv.reader(open('/Users/fmarletaz/Dropbox/Genomes/Priapulid/Analyses/Data_tables/Pcau_genes_info.txt'),delimiter='\t')}


In [ ]:
oL=[]
for l in csv.reader(open('duplicates_go_r.txt'),delimiter='\t'):
    if l[0]=='ID': continue
    gl=l[8].split('/')
    gln=[pcNm[g] for g in gl]
    glngpt=[g.split('-')[0] for g in gln if not g.startswith('Unchar')]
    print (l[2],','.join(glngpt))
    l.append(gln)
    oL.append(l)

### molecular dating of duplication : step 1 select relevant gene families ... 

In [ ]:
sel_fams=

In [16]:
print(rTrees['OG_10010'])


         /-Ampque_Aqu2.1.40812
      /-|
   /-|   \-Ampque_Aqu2.1.40817
  |  |
  |   \-Osccar_g7279
  |
  |      /-Rhoesc_Rescu_001675
  |   /-|
  |  |   \-Nemvec_NVEC200_006860_1
--|  |
  |  |            /-Musmus_ENSMUSG00000044629
  |  |         /-|
  |  |      /-|   \-Lepocu_ENSLOCG00000016014
  |  |     |  |
  |  |   /-|   \-Braflo_BL010361
  |  |  |  |
   \-|  |   \-Sackow_Sakowv30046304m
     |  |
     |  |         /-Acagra_Acagra_g2618
     |  |      /-|
     |  |     |  |   /-Pecmax_LOC117324542
     |  |     |   \-|
     |  |     |      \-Lotgig_LotgiG205111
     |  |   /-|
     |  |  |  |         /-Captel_CapteG221104
     |  |  |  |      /-|
      \-|  |  |   /-|   \-Owefus_OFUSG22021
        |  |  |  |  |
        |  |   \-|   \-Notgen_g20875
        |  |     |
        |  |     |   /-Linana_LANA07231
        |  |      \-|
        |  |         \-Phoaur_g178
        |  |
        |  |            /-Halspi_ScKMIYO_1.g1995
        |  |         /-|
        |  |      /-|   \-Pricau

## Master annotation table

In [39]:
with open('Orthogroups_stats.tsv','w') as out:
    out.write("FID\tnb_genes\tnb_species\torigin\tlost_sp\tlost_taxa\texpanded_tax\n")
    for fid,gbs in Fams.items():
        nspec,ndesc,oritax,lost=idGainLoss[fid]
        ngn=sum([len(gl) for gl in gbs.values()])
        exp=[s for s,p in famEnrich[fid].items() if p[0]]
        #rpvals=zip(pvals.keys(),[round(-np.log10(l[1]),2) for l in pvals.values()])
        outL=[fid,str(ngn),str(nspec),oritax,str(ndesc-nspec),','.join(lost),','.join(exp)]
        out.write('\t'.join(outL)+'\n')

In [ ]:
df.N.str.contains('A')  

In [14]:
import pandas as pd

In [15]:
df = pd.read_csv('Leri_ah2p_models_infos.txt',sep='\t')

In [94]:
df.PFAM_name.str.contains('Aminotran_5')

0          NaN
1          NaN
2        False
3        False
4        False
         ...  
30484    False
30485      NaN
30486      NaN
30487    False
30488      NaN
Name: PFAM_name, Length: 30489, dtype: object

,gene,Name,Desc,PFAM,PFAM_name,GO,Max_FPKM,Max_Sample,Age,Valid,Introns,Positions
0,LE00001,UnkProt_1,NaN,NaN,NaN,NaN,0.126034,Midbrain,NaN,False,1.0,Leri_713S:60462-60846
1,LE00002,UnkProt_2,NaN,NaN,NaN,NaN,5.126109,S20_24,Elasmobranchia,True,4.0,Leri_713S:27642-46600
2,LE00003,Apol3-5,apolipoprotein L3,PF05461,ApoL,GO:0006869;GO:0005576;GO:0042157;GO:0008289,0.182285,S31_33,Deuterostomia,True,2.0,Leri_81S:69654-74987
3,LE00004,Dhx32-1,DEAH-box helicase 32 (putative),PF07717;PF04408,OB_NTP_bind;HA2,GO:0004386,13.168772,Testis,Deuterostomia,True,12.0,Leri_81S:237810-330049
4,LE00005,Il12b-2,interleukin 12B,PF10420,IL12p40_C,NaN,0.729118,Intestine,Gnathostomata,True,8.0,Leri_81S:116736-149711
...,...,...,...,...,...,...,...,...,...,...,...,...
30484,LE30485,Tprg1l-2,tumor protein p63 regulated 1 like,PF12456,hSac2,NaN,184.753653,Gills,Deuterostomia,True,5.0,Leri_441S:92361-103840
30485,LE30486,UnkProt_10560,NaN,NaN,NaN,NaN,0.336184,S18_19,NaN,False,2.0,Leri_441S:73507-75265
30486,LE30487,UnkProt_10561,NaN,NaN,NaN,NaN,56.702614,S18_19,NaN,True,5.0,Leri_441S:107447-117081
30487,LE30488,Adgre1-9,adhesion G protein-coupled receptor E1,PF01825;PF16489;PF00002;PF07645,GPS;GAIN;7tm_2;EGF_CA,GO:0004930;GO:0007186;GO:0016021;GO:0005509,0.900612,Intestine,Deuterostomia,True,17.0,Leri_289S:74553-128615


In [9]:
df[df['Valid']==True]

NameError: name 'df' is not defined

### Age of MFUZZ/WGCNA cluster

In [10]:
from collections import defaultdict 

In [59]:
from scipy.stats import hypergeom
import pandas as pd

In [5]:
pdclst={l.strip().split()[0]:l.strip().split()[1] for l in open("../mfuzz/priap_clusters_run1_14.tsv") if len(l.strip().split())==2}

In [27]:
clst=list(set(list(pdclst.values())))

In [ ]:
32607

In [7]:
pdst={l.strip().split()[0]:l.strip().split()[1] for l in open("Pcau_phylostrata.txt")}

In [17]:
tbc=defaultdict(int)
for g in pdclst:
    if g in pdst:
        e=pdst[g]
        tbc[e]+=1
    else:
        tbc['Pcau']+=1

In [20]:
tbc

defaultdict(int,
            {'Metazoa': 19455,
             'Pcau': 7857,
             'Eumetazoa': 2703,
             'Bilateria': 1377,
             'Priapulida': 608,
             'Protostomia': 486,
             'Ecdysozoa': 121})

In [ ]:
pdclst

In [ ]:
M=26054
n=tbc['Ambulacraria']
N=len()
stats.scipy.stats.hypergeom(M, n, N)

In [ ]:
hypres=[]
for m,e in tbc.items():
    pv=[]
    for c in clst:
        gic=[g for g,k in pdclst.items() if k==c]
        mic=[g for g in gic if pdst.get(g,'Pcau')==m]
        
        #hpd=hypergeom(32607, tbc[m],len(gic))
        pt=hypergeom.sf(len(mic)-1,32607,tbc[m],len(gic))
        # p_value = hypergeom.sf(k-1, N, K, n)
        expect = (len(gic) * tbc[m]) / 32607
        folden = len(mic) / expect if expect > 0 else float('inf')
        print(m,c,len(gic),tbc[m],len(mic),folden,pt)
        hypres.append([m,c,len(gic),tbc[m],len(mic),folden,pt])
    #print(m,pv)

    


In [60]:
restret=pd.DataFrame(hypres, columns = [['strat','clust','nclust','nstrat','ovl','enfold','pval']])  

In [61]:
restret

,strat,clust,nclust,nstrat,ovl,enfold,pval
0,Metazoa,14,3030,19455,1838,1.016676,1.244311e-01
1,Metazoa,1,1544,19455,777,0.843438,1.000000e+00
2,Metazoa,13,2237,19455,1530,1.146318,5.850701e-19
3,Metazoa,2,1509,19455,973,1.080695,4.763437e-05
4,Metazoa,11,2416,19455,1227,0.851191,1.000000e+00
...,...,...,...,...,...,...,...
93,Ecdysozoa,7,1632,121,6,0.990733,5.679246e-01
94,Ecdysozoa,12,3794,121,13,0.923361,6.623008e-01
95,Ecdysozoa,4,2855,121,8,0.755108,8.412579e-01
96,Ecdysozoa,9,4336,121,14,0.870090,7.509671e-01


In [62]:
   restret.to_csv("mfuzz_strat.tsv",sep='\t',index=False)